In [10]:
# Importe les outils de préparation des données et charge le jeu d'entraînement.
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
df=pd.read_csv("../data/raw/aug_train.csv")

In [3]:
# Définit les variables explicatives et la cible, en excluant l'identifiant du candidat.
X = df.drop(columns=["target", "enrollee_id"])
y = df["target"].astype(int)

In [4]:
# Crée des ensembles d'entraînement et de test en conservant la proportion des classes.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [5]:
# Convertit les valeurs textuelles extrêmes de l'expérience en nombres, puis vérifie le résultat.
def convertir_experience(s):
    return pd.to_numeric(s.replace({"<1": "0", ">20": "21"}))

exp = convertir_experience(X_train["experience"])
print(exp.dtype, exp.isna().sum(), exp.min(), exp.max())

float64 50 0.0 21.0


In [6]:
# Convertit les modalités du délai depuis le dernier emploi en nombres, puis vérifie le résultat.
def convertir_last_new_job(s):
    return pd.to_numeric(s.replace({"never": "0", ">4": "5"}))

lnj = convertir_last_new_job(X_train["last_new_job"])
print(lnj.dtype, lnj.isna().sum(), lnj.min(), lnj.max())

float64 343 0.0 5.0


In [7]:
# Examine le nombre de valeurs manquantes et le type de chaque variable de l'ensemble d'entraînement.
print(X_train.isna().sum())
print(X_train.dtypes)


city                         0
city_development_index       0
gender                    3574
relevent_experience          0
enrolled_university        317
education_level            362
major_discipline          2258
experience                  50
company_size              4714
company_type              4881
last_new_job               343
training_hours               0
dtype: int64
city                          str
city_development_index    float64
gender                        str
relevent_experience           str
enrolled_university           str
education_level               str
major_discipline              str
experience                    str
company_size                  str
company_type                  str
last_new_job                  str
training_hours              int64
dtype: object


In [8]:
# Convertit et impute les variables numériques avec la médiane calculée sur l'ensemble d'entraînement.
def convertir_numeriques(df_):
    df_ = df_.copy()
    df_["experience"] = convertir_experience(df_["experience"])
    df_["last_new_job"] = convertir_last_new_job(df_["last_new_job"])
    return df_

branche_num = Pipeline([
    ("conversion", FunctionTransformer(convertir_numeriques)),
    ("imputation", SimpleImputer(strategy="median")),
])

cols_g1 = ["experience", "last_new_job"]

Xtr = branche_num.fit_transform(X_train[cols_g1])
Xte = branche_num.transform(X_test[cols_g1])

print(Xtr.shape, Xte.shape)
print(np.isnan(Xtr).sum(), np.isnan(Xte).sum())
print(branche_num.named_steps["imputation"].statistics_)

(15326, 2) (3832, 2)
0 0
[9. 1.]


In [11]:
# Remplace les catégories manquantes par « Inconnu », puis les encode en variables indicatrices.
cols_inconnu = ["company_size", "company_type", "gender", "education_level"]

branche_inconnu = Pipeline([
    ("imputation", SimpleImputer(strategy="constant", fill_value="Inconnu")),
    ("encodage", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

Xtr = branche_inconnu.fit_transform(X_train[cols_inconnu])
Xte = branche_inconnu.transform(X_test[cols_inconnu])

print(Xtr.shape, Xte.shape)
print(branche_inconnu.named_steps["encodage"].categories_)

(15326, 26) (3832, 26)
[array(['10/49', '100-500', '1000-4999', '10000+', '50-99', '500-999',
       '5000-9999', '<10', 'Inconnu'], dtype=object), array(['Early Stage Startup', 'Funded Startup', 'Inconnu', 'NGO', 'Other',
       'Public Sector', 'Pvt Ltd'], dtype=object), array(['Female', 'Inconnu', 'Male', 'Other'], dtype=object), array(['Graduate', 'High School', 'Inconnu', 'Masters', 'Phd',
       'Primary School'], dtype=object)]
